# Appendix 10.6: Structured Outputs

- [Lesson](#lesson)
- [Exercise](#exercise)
- [Example Playground](#example-playground)

## Setup

Run the following setup cell to load your API key and establish the `extract` helper function.

In [ ]:
!pip install anthropic

import anthropic, json

%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def extract(prompt, schema_tool):
    """Force Claude to respond via a single tool call matching schema_tool, and return the parsed input dict."""
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1000,
        temperature=0.0,
        tools=[schema_tool],
        tool_choice={"type": "tool", "name": schema_tool["name"]},
        messages=[{"role": "user", "content": prompt}],
    )
    tool_use_block = next(b for b in response.content if b.type == "tool_use")
    return tool_use_block.input

---

## Lesson

Back in Chapter 5, you learned to prefill `{` to nudge Claude toward JSON output. That works, but it's a nudge, not a guarantee — occasionally Claude adds a stray sentence, an unescaped quote, or a trailing comma, and `json.loads()` throws on 1 request out of a few hundred. Fine for a demo notebook; not fine for a production pipeline.

**The reliable fix, which you already have all the pieces for from Appendix 10.2:** define a tool whose `input_schema` *is* the exact shape of data you want, and force Claude to call it with `tool_choice={"type": "tool", "name": ...}`. Claude was never actually going to "run" this tool — you're repurposing tool use purely as a schema-constrained output channel. The `input` on the returned `tool_use` block is already a parsed dict/list/number matching your schema — no string parsing, no malformed-JSON exceptions, no prompt-engineering the words "respond only in valid JSON" and hoping.

**Where this shows up in real systems:**
- Turning a freeform customer email into a structured support ticket (category, urgency, customer sentiment) that gets inserted straight into a database row.
- An agent's decision step, where the *next action* needs to come back as a typed object (`{"action": "refund", "amount": 42.50}`) that downstream code branches on.
- Extracting line items from an invoice or receipt into a list of structured records.
- **The 2 AM production bug this prevents:** a pipeline calls `json.loads(claude_response)` on every request; under load, a small percentage of responses come back with a leading "Sure, here's the JSON:" or a subtly malformed structure, and the whole batch job crashes on that one row hours into a run. Schema-forced tool use eliminates this entire failure class, because the API itself enforces the schema instead of you trusting the model's plain-text formatting.

**Version note:** this pattern has been available since native tool use shipped, and remains the documented, portable way to get schema-guaranteed structured output from Claude across model versions. It also composes cleanly with everything else in this course — you can force one tool for guaranteed output *shape* while still applying every prompting technique you've learned to control the *content*.

### Example — extracting a structured ticket from a support email

In [ ]:
ticket_schema = {
    "name": "record_support_ticket",
    "description": "Records a structured support ticket extracted from a customer email.",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_name": {"type": "string"},
            "category": {
                "type": "string",
                "enum": ["billing", "technical", "shipping", "other"]
            },
            "urgency": {
                "type": "integer",
                "description": "1 (low) to 5 (critical)",
                "minimum": 1,
                "maximum": 5
            },
            "summary": {"type": "string", "description": "One-sentence summary of the issue."}
        },
        "required": ["customer_name", "category", "urgency", "summary"]
    }
}

email = (
    "Hi, this is Marcus Webb. My package was supposed to arrive 5 days ago and tracking hasn't updated "
    "since. I have a work event tomorrow that needs what's in this box. Please help ASAP!"
)

ticket = extract(f"Extract a support ticket from this email:\n\n{email}", ticket_schema)
print(json.dumps(ticket, indent=2))
print(type(ticket))

Notice `ticket` is already a real Python `dict` with the exact keys and types you specified — `urgency` is an `int`, `category` is guaranteed to be one of your four enum values. No parsing step, no validation step; the schema *is* the contract.

---

## Exercise

### Exercise 10.6.1 - Structured Product Review

Write a schema tool `review_schema` that extracts, from a freeform product review, a `product_name` (string), a `rating` (integer 1-5), a `sentiment` (one of `"positive"`, `"neutral"`, `"negative"`), and `key_phrases` (an array of short strings capturing the reviewer's main points).

In [ ]:
# Your schema goes here — this is the only field you should change
review_schema = {
    "name": "record_review",
    "description": "[Replace this text]",
    "input_schema": {
        "type": "object",
        "properties": {
            # [Replace this comment with your properties]
        },
        "required": []
    }
}

review = (
    "I bought the Aria wireless headphones last month. The sound quality is fantastic and the battery "
    "lasts all week, but the ear cups get uncomfortable after about two hours. Overall pretty happy though, "
    "I'd say a solid 4 out of 5."
)

result = extract(f"Extract structured data from this review:\n\n{review}", review_schema)
print(json.dumps(result, indent=2))

print("\n--------------------------- GRADING ---------------------------")
checks = [
    result.get("rating") == 4,
    result.get("sentiment") == "positive",
    isinstance(result.get("key_phrases"), list) and len(result.get("key_phrases", [])) >= 2,
]
print("All checks passed!" if all(checks) else f"Checks: {checks}")

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_10_6_1_hint; print(exercise_10_6_1_hint)

### Congrats!

**Recap:** for any output your code is going to parse automatically, prefer schema-forced tool use over prefill-and-hope. Define an `input_schema` that matches the shape you want, force it with `tool_choice`, and read `tool_use_block.input` directly — no JSON parsing, no malformed-output edge cases.

**Interview-answer framing:** "For guaranteed-structure output, I don't rely on asking the model to 'output valid JSON' — I define a tool whose input schema matches exactly the object shape I need, and force its use with `tool_choice`. The API then returns a `tool_use` block whose `input` is already schema-conforming, parsed data — not a plain-text string I need to validate and parse myself. It's the same tool-use mechanism used for real function calling, just repurposed as a structured-output contract."

Head over to Appendix 10.7 to learn about citations.

---

## Example Playground

This is an area for you to experiment freely with the prompt examples shown in this lesson.

In [ ]:
ticket_schema = {
    "name": "record_support_ticket",
    "description": "Records a structured support ticket extracted from a customer email.",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_name": {"type": "string"},
            "category": {"type": "string", "enum": ["billing", "technical", "shipping", "other"]},
            "urgency": {"type": "integer", "minimum": 1, "maximum": 5},
            "summary": {"type": "string"}
        },
        "required": ["customer_name", "category", "urgency", "summary"]
    }
}

email = (
    "Hi, this is Marcus Webb. My package was supposed to arrive 5 days ago and tracking hasn't updated "
    "since. I have a work event tomorrow that needs what's in this box. Please help ASAP!"
)

ticket = extract(f"Extract a support ticket from this email:\n\n{email}", ticket_schema)
print(json.dumps(ticket, indent=2))